# InsectDex - Kaggle Auto Fine-tune YOLO

Notebook này dùng cho Kaggle Scheduled Notebook hằng ngày.

Luồng xử lý:

1. Kết nối Supabase bằng Kaggle Secrets.
2. Tìm `dataset_versions.status = 'exported'` mới nhất.
3. Tải YOLO dataset mới từ Supabase Storage.
4. Lấy một phần dataset cũ đang attach trong Kaggle Input.
5. Merge dataset cũ + dataset mới, remap class id theo `data.yaml` hợp nhất.
6. Tải model production hiện tại từ Hugging Face nếu có, nếu không thì fallback sang YOLO pretrained.
7. Fine-tune YOLO.
8. Upload model candidate lên Hugging Face hoặc Supabase Storage.
9. Ghi `model_versions` vào Supabase.
10. Cập nhật `dataset_versions.status = 'used_for_training'` để lần schedule sau không train lại.

Giai đoạn demo hiện tại có thể dùng dataset mới rất nhỏ, ví dụ 2 ảnh. Notebook sẽ train ở chế độ demo và **không tự promote production** nếu `AUTO_PROMOTE = False`.


In [1]:
# ============================================================
# 1. Install dependencies
# ============================================================

%pip install -q --upgrade --force-reinstall "pillow>=10.4.0,<12.0.0"
%pip install -q --upgrade "ultralytics>=8.3.0" "huggingface_hub" "supabase" "pyyaml"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 42.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.6 MB/s eta 0:00:00
E

In [2]:
# ============================================================
# 2. Imports
# ============================================================

import os
import json
import yaml
import uuid
import shutil
import zipfile
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

from tqdm import tqdm
from PIL import Image
from supabase import create_client
from ultralytics import YOLO
from huggingface_hub import hf_hub_download, HfApi

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# ============================================================
# 3. Config
# ============================================================

# Kaggle Secrets cần tạo:
# - SUPABASE_URL
# - SUPABASE_SERVICE_ROLE_KEY
# - HF_TOKEN                       optional nếu chỉ train local, bắt buộc nếu tải/upload model HF private
# - HF_MODEL_REPO                  ví dụ: TOILAXIEN/insectdex-yolo-models
# - HF_PRODUCTION_MODEL_PATH       optional, ví dụ: production/best.pt

# Dataset cũ trên Kaggle Input. Nếu để AUTO, notebook sẽ tự tìm data.yaml trong /kaggle/input.
BASE_DATASET_DIR = "AUTO"

# Training output
WORK_DIR = Path("/kaggle/working/insectdex_auto_finetune")
NEW_DATASET_DIR = WORK_DIR / "new_dataset"
BASE_SAMPLE_DIR = WORK_DIR / "base_sample_dataset"
MERGED_DATASET_DIR = WORK_DIR / "merged_dataset"
MODEL_OUTPUT_DIR = WORK_DIR / "runs"
DOWNLOAD_DIR = WORK_DIR / "downloads"

# Sampling dữ liệu cũ để giảm quên class cũ.
# Demo có thể để 20. Chạy thật có thể tăng 50-100 nếu đủ thời gian GPU.
BASE_TRAIN_IMAGES_PER_CLASS = 20
BASE_VAL_IMAGES_PER_CLASS = 5

# YOLO training config.
DEMO_MODE = True
EPOCHS_DEMO = 1
EPOCHS_PRODUCTION = 30
IMG_SIZE = 640
BATCH_SIZE = 8
PATIENCE = 5

# Auto promote production: hiện tại nên False, vì dataset mới demo chỉ 2 ảnh.
AUTO_PROMOTE = False

# Nếu dataset mới quá nhỏ thì vẫn train demo, nhưng không production.
MIN_NEW_IMAGES_FOR_REAL_TRAIN = 20

# Hugging Face artifact paths.
HF_CANDIDATE_PREFIX = "candidates"
HF_PRODUCTION_PATH = "production/best.pt"
HF_PRODUCTION_META_PATH = "production/model_meta.json"
HF_PRODUCTION_DATA_YAML_PATH = "production/data.yaml"

# Supabase fallback model storage, nếu không cấu hình HF_MODEL_REPO/HF_TOKEN.
MODEL_STORAGE_BUCKET = "observations"
MODEL_STORAGE_PREFIX = "models/yolo_candidates"

# Nếu không tải được model production từ HF, fallback sang pretrained.
FALLBACK_PRETRAINED_MODEL = "yolo11n.pt"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


In [4]:
# ============================================================
# 4. Secrets helpers
# ============================================================

def get_secret(name: str, default: Optional[str] = None, required: bool = False) -> Optional[str]:
    value = os.environ.get(name)
    if value:
        return value

    if UserSecretsClient is not None:
        try:
            value = UserSecretsClient().get_secret(name)
            if value:
                return value
        except Exception:
            pass

    if required:
        raise RuntimeError(f"Missing required secret: {name}")
    return default

SUPABASE_URL = get_secret("SUPABASE_URL", required=True)
SUPABASE_SERVICE_ROLE_KEY = get_secret("SUPABASE_SERVICE_ROLE_KEY", required=True)
HF_TOKEN = get_secret("HF_TOKEN", default=None)
HF_MODEL_REPO = get_secret("HF_MODEL_REPO", default=None)
HF_PRODUCTION_MODEL_PATH = get_secret("HF_PRODUCTION_MODEL_PATH", default=HF_PRODUCTION_PATH)

print("SUPABASE_URL:", SUPABASE_URL)
print("HF_MODEL_REPO:", HF_MODEL_REPO)
print("HF production model path:", HF_PRODUCTION_MODEL_PATH)
print("HF token available:", bool(HF_TOKEN))


SUPABASE_URL: https://wptbgevxqtjnmpqkerkl.supabase.co
HF_MODEL_REPO: TOILAXIEN/insectdex-yolo-models
HF production model path: production/best.pt
HF token available: True


In [5]:
# ============================================================
# 5. Clients and utility functions
# ============================================================

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
hf_api = HfApi(token=HF_TOKEN) if HF_TOKEN else None

WORK_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def file_sha256(path: Path) -> str:
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(block)
    return sha.hexdigest()


def safe_rmtree(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)


def load_yaml(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_yaml(data: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)


def normalize_name(name: Any) -> str:
    return str(name).strip()


def get_names_from_data_yaml(data_yaml: dict) -> List[str]:
    names = data_yaml.get("names")
    if isinstance(names, dict):
        return [str(names[i]) for i in sorted(names.keys(), key=lambda x: int(x))]
    if isinstance(names, list):
        return [str(x) for x in names]
    raise ValueError("data.yaml missing valid names")


def list_image_files(folder: Path) -> List[Path]:
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    if not folder.exists():
        return []
    return [p for p in folder.rglob("*") if p.suffix.lower() in exts]


def label_path_for_image(image_path: Path, images_root: Path, labels_root: Path) -> Path:
    rel = image_path.relative_to(images_root)
    return (labels_root / rel).with_suffix(".txt")


def copy_image_and_remap_label(
    src_image: Path,
    src_label: Path,
    dst_image: Path,
    dst_label: Path,
    class_id_map: Dict[int, int],
) -> bool:
    if not src_label.exists():
        return False

    dst_image.parent.mkdir(parents=True, exist_ok=True)
    dst_label.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src_image, dst_image)

    remapped_lines = []
    with open(src_label, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            old_id = int(float(parts[0]))
            if old_id not in class_id_map:
                continue
            parts[0] = str(class_id_map[old_id])
            remapped_lines.append(" ".join(parts[:5]))

    if not remapped_lines:
        return False

    with open(dst_label, "w", encoding="utf-8") as f:
        f.write("".join(remapped_lines) + "")

    return True


In [6]:
# ============================================================
# 6. Find exported dataset_versions from Supabase
# ============================================================

def find_next_exported_dataset() -> Optional[dict]:
    # Lấy dataset mới nhất đã export nhưng chưa dùng train.
    response = (
        supabase.table("dataset_versions")
        .select("*")
        .eq("status", "exported")
        .order("created_at", desc=True)
        .limit(1)
        .execute()
    )
    rows = response.data or []
    if not rows:
        return None
    return rows[0]


dataset_version = find_next_exported_dataset()

if not dataset_version:
    print("No dataset_versions.status = 'exported'. Nothing to train today.")
    TRAINING_SHOULD_RUN = False
else:
    TRAINING_SHOULD_RUN = True
    print("Selected dataset_version:")
    print(json.dumps(dataset_version, indent=2, ensure_ascii=False, default=str))


Selected dataset_version:
{
  "id": "ea6317a0-abb6-43fb-b795-db622a81cf10",
  "name": "insectdex_yolo_20260605_010113_49aee52e",
  "class_count": 1,
  "image_count": 2,
  "source_filter": {
    "source": "databricks_export",
    "mlops_status": "approved_for_training",
    "bbox_required": true,
    "source_bucket": "observations",
    "dataset_prefix": "datasets/yolo_exports",
    "auto_label_status": "accepted",
    "min_images_per_class": 2
  },
  "yaml_path": "data.yaml",
  "dataset_zip_path": "datasets/yolo_exports/insectdex_yolo_20260605_010113_49aee52e.zip",
  "hash": "0729236d843d7fe122637ec475f45e5bdca4b6883e772f57ef9d665cf5feb0fd",
  "status": "exported",
  "created_at": "2026-06-05T01:01:18.279092+00:00",
  "class_ids": [
    "cicada"
  ],
  "train_count": 1,
  "val_count": 1,
  "storage_bucket": "observations",
  "storage_path": "datasets/yolo_exports/insectdex_yolo_20260605_010113_49aee52e.zip",
  "export_config": {
    "names": [
      "Cicada"
    ],
    "format": "yolo"

In [7]:
# ============================================================
# 7. Download new YOLO dataset from Supabase Storage
# ============================================================

def download_dataset_zip(dataset_version: dict) -> Path:
    bucket = dataset_version.get("storage_bucket") or "observations"
    storage_path = (
        dataset_version.get("storage_path")
        or dataset_version.get("dataset_zip_path")
    )
    if not storage_path:
        raise RuntimeError("dataset_versions missing storage_path/dataset_zip_path")

    zip_path = DOWNLOAD_DIR / Path(storage_path).name
    print(f"Downloading dataset zip from Supabase: bucket={bucket}, path={storage_path}")
    data = supabase.storage.from_(bucket).download(storage_path)
    with open(zip_path, "wb") as f:
        f.write(data)

    if zip_path.stat().st_size == 0:
        raise RuntimeError("Downloaded dataset zip is empty")

    print("Downloaded:", zip_path, "size=", zip_path.stat().st_size)
    return zip_path


def unzip_dataset(zip_path: Path, dst_dir: Path) -> None:
    safe_rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dst_dir)
    print("Unzipped dataset to:", dst_dir)


if TRAINING_SHOULD_RUN:
    dataset_zip_path = download_dataset_zip(dataset_version)
    unzip_dataset(dataset_zip_path, NEW_DATASET_DIR)

    new_data_yaml_path = NEW_DATASET_DIR / "data.yaml"
    if not new_data_yaml_path.exists():
        # Một số zip có thể chứa folder root. Tìm data.yaml bên trong.
        candidates = list(NEW_DATASET_DIR.rglob("data.yaml"))
        if not candidates:
            raise RuntimeError("New dataset zip does not contain data.yaml")
        new_root = candidates[0].parent
        print("Detected nested new dataset root:", new_root)
        NEW_DATASET_DIR = new_root
        new_data_yaml_path = candidates[0]

    new_data_yaml = load_yaml(new_data_yaml_path)
    new_names = get_names_from_data_yaml(new_data_yaml)
    print("New dataset names:", new_names)


Downloaded: /kaggle/working/insectdex_auto_finetune/downloads/insectdex_yolo_20260605_010113_49aee52e.zip size= 16738
Unzipped dataset to: /kaggle/working/insectdex_auto_finetune/new_dataset
New dataset names: ['Cicada']


In [8]:
# ============================================================
# 8. Locate old/base YOLO dataset in Kaggle Input
# ============================================================

def find_base_dataset_dir() -> Path:
    if BASE_DATASET_DIR != "AUTO":
        p = Path(BASE_DATASET_DIR)
        if not (p / "data.yaml").exists():
            raise RuntimeError(f"BASE_DATASET_DIR does not contain data.yaml: {p}")
        return p

    candidates = list(Path("/kaggle/input").rglob("data.yaml"))
    if not candidates:
        raise RuntimeError("Cannot find old dataset data.yaml in /kaggle/input. Attach your old YOLO dataset to this Kaggle notebook.")

    # Ưu tiên dataset có images/train và labels/train.
    scored = []
    for c in candidates:
        root = c.parent
        score = 0
        if (root / "images" / "train").exists(): score += 10
        if (root / "labels" / "train").exists(): score += 10
        if (root / "images" / "val").exists() or (root / "images" / "valid").exists() or (root / "images" / "test").exists(): score += 5
        scored.append((score, root, c))

    scored.sort(reverse=True, key=lambda x: x[0])
    root = scored[0][1]
    print("Detected base dataset root:", root)
    return root


if TRAINING_SHOULD_RUN:
    BASE_ROOT = find_base_dataset_dir()
    base_data_yaml = load_yaml(BASE_ROOT / "data.yaml")
    base_names = get_names_from_data_yaml(base_data_yaml)
    print("Base dataset names:", base_names)


Detected base dataset root: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo
Base dataset names: ['ant', 'butterfly', 'cockroach', 'dragonfly', 'fly', 'grasshopper', 'honeybee', 'ladybug', 'mosquito', 'spider']


In [9]:
# ============================================================
# 9. Build merged class names and sample old dataset
# ============================================================

def build_unified_names(base_names: List[str], new_names: List[str]) -> List[str]:
    unified = []
    seen = set()
    for name in base_names + new_names:
        clean = normalize_name(name)
        key = clean.lower()
        if key not in seen:
            unified.append(clean)
            seen.add(key)
    return unified


def make_class_id_map(source_names: List[str], unified_names: List[str]) -> Dict[int, int]:
    unified_lookup = {name.lower(): idx for idx, name in enumerate(unified_names)}
    mapping = {}
    for old_idx, name in enumerate(source_names):
        key = normalize_name(name).lower()
        if key not in unified_lookup:
            raise RuntimeError(f"Class name not found in unified names: {name}")
        mapping[old_idx] = unified_lookup[key]
    return mapping


def get_split_roots(root: Path, split: str) -> Tuple[Path, Path]:
    image_root = root / "images" / split
    label_root = root / "labels" / split
    return image_root, label_root


def sample_base_images_by_class(root: Path, names: List[str], split: str, per_class: int) -> List[Path]:
    image_root, label_root = get_split_roots(root, split)
    if not image_root.exists() or not label_root.exists():
        return []

    images_by_class: Dict[int, List[Path]] = {i: [] for i in range(len(names))}

    for img_path in list_image_files(image_root):
        lbl_path = label_path_for_image(img_path, image_root, label_root)
        if not lbl_path.exists():
            continue
        class_ids = set()
        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    try:
                        class_ids.add(int(float(parts[0])))
                    except Exception:
                        pass
        for cid in class_ids:
            if cid in images_by_class:
                images_by_class[cid].append(img_path)

    selected = []
    for cid, imgs in images_by_class.items():
        random.shuffle(imgs)
        take = imgs[:per_class]
        selected.extend(take)
        print(f"Base sample {split} class {cid} ({names[cid]}): {len(take)} image(s)")

    # Tránh copy trùng một ảnh nếu ảnh có nhiều class.
    selected_unique = list(dict.fromkeys(selected))
    return selected_unique


if TRAINING_SHOULD_RUN:
    unified_names = build_unified_names(base_names, new_names)
    base_id_map = make_class_id_map(base_names, unified_names)
    new_id_map = make_class_id_map(new_names, unified_names)

    print("Unified names:", unified_names)
    print("Base class id map:", base_id_map)
    print("New class id map:", new_id_map)


Unified names: ['ant', 'butterfly', 'cockroach', 'dragonfly', 'fly', 'grasshopper', 'honeybee', 'ladybug', 'mosquito', 'spider', 'Cicada']
Base class id map: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}
New class id map: {0: 10}


In [10]:
# ============================================================
# 10. Merge old sampled dataset + new dataset
# ============================================================

def copy_dataset_split(
    src_root: Path,
    split: str,
    dst_root: Path,
    prefix: str,
    class_id_map: Dict[int, int],
    selected_images: Optional[List[Path]] = None,
) -> int:
    src_image_root, src_label_root = get_split_roots(src_root, split)
    if not src_image_root.exists() or not src_label_root.exists():
        print(f"Skip missing split: {src_root} {split}")
        return 0

    if selected_images is None:
        selected_images = list_image_files(src_image_root)

    copied = 0
    for src_img in tqdm(selected_images, desc=f"copy {prefix}/{split}"):
        src_lbl = label_path_for_image(src_img, src_image_root, src_label_root)
        if not src_lbl.exists():
            continue

        rel = src_img.relative_to(src_image_root)
        safe_name = f"{prefix}_{rel.as_posix().replace('/', '_')}"
        dst_img = dst_root / "images" / split / safe_name
        dst_lbl = (dst_root / "labels" / split / safe_name).with_suffix(".txt")

        ok = copy_image_and_remap_label(src_img, src_lbl, dst_img, dst_lbl, class_id_map)
        if ok:
            copied += 1
    return copied


def detect_split(root: Path, preferred: str) -> str:
    """
    YOLO datasets may use val, valid, or test.
    Return the first existing split compatible with the preferred split.
    """
    if (root / "images" / preferred).exists():
        return preferred

    if preferred in {"val", "valid", "test"}:
        for alt in ["val", "valid", "test"]:
            if (root / "images" / alt).exists():
                return alt

    return preferred


if TRAINING_SHOULD_RUN:
    safe_rmtree(MERGED_DATASET_DIR)
    for split in ["train", "val"]:
        (MERGED_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (MERGED_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    # Sample old dataset.
    base_train_selected = sample_base_images_by_class(BASE_ROOT, base_names, "train", BASE_TRAIN_IMAGES_PER_CLASS)
    base_val_split = detect_split(BASE_ROOT, "val")
    base_val_selected = sample_base_images_by_class(BASE_ROOT, base_names, base_val_split, BASE_VAL_IMAGES_PER_CLASS)

    base_train_count = copy_dataset_split(BASE_ROOT, "train", MERGED_DATASET_DIR, "base", base_id_map, base_train_selected)
    base_val_count = copy_dataset_split(BASE_ROOT, base_val_split, MERGED_DATASET_DIR, "base", base_id_map, base_val_selected)

    # Copy all new dataset images.
    new_train_split = detect_split(NEW_DATASET_DIR, "train")
    new_val_split = detect_split(NEW_DATASET_DIR, "val")
    new_train_count = copy_dataset_split(NEW_DATASET_DIR, new_train_split, MERGED_DATASET_DIR, "new", new_id_map, None)
    new_val_count = copy_dataset_split(NEW_DATASET_DIR, new_val_split, MERGED_DATASET_DIR, "new", new_id_map, None)

    merged_train_count = len(list_image_files(MERGED_DATASET_DIR / "images" / "train"))
    merged_val_count = len(list_image_files(MERGED_DATASET_DIR / "images" / "val"))

    merged_data_yaml = {
        "path": str(MERGED_DATASET_DIR),
        "train": "images/train",
        "val": "images/val",
        "nc": len(unified_names),
        "names": unified_names,
    }
    save_yaml(merged_data_yaml, MERGED_DATASET_DIR / "data.yaml")

    print("Merge summary:")
    print("base_train_count:", base_train_count)
    print("base_val_count:", base_val_count)
    print("new_train_count:", new_train_count)
    print("new_val_count:", new_val_count)
    print("merged_train_count:", merged_train_count)
    print("merged_val_count:", merged_val_count)
    print("merged data.yaml:", MERGED_DATASET_DIR / "data.yaml")

    if merged_train_count == 0:
        raise RuntimeError("Merged dataset has no train images")
    if merged_val_count == 0:
        print("WARNING: merged dataset has no val images. YOLO may still run poorly.")


Base sample train class 0 (ant): 20 image(s)
Base sample train class 1 (butterfly): 20 image(s)
Base sample train class 2 (cockroach): 20 image(s)
Base sample train class 3 (dragonfly): 20 image(s)
Base sample train class 4 (fly): 20 image(s)
Base sample train class 5 (grasshopper): 20 image(s)
Base sample train class 6 (honeybee): 20 image(s)
Base sample train class 7 (ladybug): 20 image(s)
Base sample train class 8 (mosquito): 20 image(s)
Base sample train class 9 (spider): 20 image(s)
Base sample val class 0 (ant): 5 image(s)
Base sample val class 1 (butterfly): 5 image(s)
Base sample val class 2 (cockroach): 5 image(s)
Base sample val class 3 (dragonfly): 5 image(s)
Base sample val class 4 (fly): 5 image(s)
Base sample val class 5 (grasshopper): 5 image(s)
Base sample val class 6 (honeybee): 5 image(s)
Base sample val class 7 (ladybug): 5 image(s)
Base sample val class 8 (mosquito): 5 image(s)
Base sample val class 9 (spider): 5 image(s)


copy new/val: 100%|██████████| 1/1 [00:00<00:00, 824.35it/s]

Merge summary:
base_train_count: 200
base_val_count: 50
new_train_count: 1
new_val_count: 1
merged_train_count: 201
merged_val_count: 51
merged data.yaml: /kaggle/working/insectdex_auto_finetune/merged_dataset/data.yaml


In [11]:
# ============================================================
# 11. Download production model from Hugging Face or fallback
# ============================================================

def download_production_model() -> str:
    if HF_MODEL_REPO:
        try:
            print(f"Downloading production model from HF: {HF_MODEL_REPO}/{HF_PRODUCTION_MODEL_PATH}")
            model_path = hf_hub_download(
                repo_id=HF_MODEL_REPO,
                filename=HF_PRODUCTION_MODEL_PATH,
                repo_type="model",
                token=HF_TOKEN,
                local_dir=str(DOWNLOAD_DIR / "hf_model"),
            )
            print("Downloaded production model:", model_path)
            return model_path
        except Exception as exc:
            print("WARNING: cannot download HF production model. Fallback to pretrained.")
            print(exc)

    print("Using fallback pretrained model:", FALLBACK_PRETRAINED_MODEL)
    return FALLBACK_PRETRAINED_MODEL


if TRAINING_SHOULD_RUN:
    start_model_path = download_production_model()


production/best.pt:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

Downloaded production model: /kaggle/working/insectdex_auto_finetune/downloads/hf_model/production/best.pt


In [12]:
# ============================================================
# 12. Train / fine-tune YOLO
# ============================================================

if TRAINING_SHOULD_RUN:
    new_image_count = int(dataset_version.get("image_count") or 0)
    real_train = new_image_count >= MIN_NEW_IMAGES_FOR_REAL_TRAIN and not DEMO_MODE
    epochs = EPOCHS_PRODUCTION if real_train else EPOCHS_DEMO

    print("new_image_count:", new_image_count)
    print("DEMO_MODE:", DEMO_MODE)
    print("real_train:", real_train)
    print("epochs:", epochs)

    model = YOLO(start_model_path)

    train_result = model.train(
        data=str(MERGED_DATASET_DIR / "data.yaml"),
        epochs=epochs,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        project=str(MODEL_OUTPUT_DIR),
        name="train",
        exist_ok=True,
        seed=RANDOM_SEED,
        pretrained=True,
        verbose=True,
    )

    run_dir = MODEL_OUTPUT_DIR / "train"
    best_pt = run_dir / "weights" / "best.pt"
    last_pt = run_dir / "weights" / "last.pt"

    if not best_pt.exists():
        if last_pt.exists():
            best_pt = last_pt
        else:
            raise RuntimeError("Training finished but no best.pt/last.pt found")

    print("Best model:", best_pt)
    print("Best model size:", best_pt.stat().st_size)


new_image_count: 2
DEMO_MODE: True
real_train: False
epochs: 1
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/insectdex_auto_finetune/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/insectdex_auto_finetune/downloads/hf_model/production

In [13]:
# ============================================================
# 13. Collect metrics and metadata
# ============================================================

def read_results_csv(run_dir: Path) -> dict:
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return {}
    try:
        import pandas as pd
        df = pd.read_csv(results_csv)
        if df.empty:
            return {}
        last = df.iloc[-1].to_dict()
        return {str(k).strip(): (float(v) if isinstance(v, (int, float)) else str(v)) for k, v in last.items()}
    except Exception as exc:
        print("WARNING: cannot parse results.csv", exc)
        return {}


if TRAINING_SHOULD_RUN:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    model_name = f"insectdex_yolo_candidate_{timestamp}"
    model_version = timestamp
    model_hash = file_sha256(best_pt)
    metrics_json = read_results_csv(run_dir)

    model_meta = {
        "model_name": model_name,
        "model_version": model_version,
        "created_at": now_utc_iso(),
        "dataset_version_id": dataset_version["id"],
        "dataset_name": dataset_version.get("name"),
        "base_dataset_root": str(BASE_ROOT),
        "start_model_path": str(start_model_path),
        "unified_names": unified_names,
        "nc": len(unified_names),
        "demo_mode": DEMO_MODE,
        "auto_promote": AUTO_PROMOTE,
        "epochs": epochs,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "new_image_count": new_image_count,
        "merged_train_count": merged_train_count,
        "merged_val_count": merged_val_count,
        "model_sha256": model_hash,
        "metrics": metrics_json,
    }

    meta_path = run_dir / "model_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(model_meta, f, ensure_ascii=False, indent=2)

    print(json.dumps(model_meta, ensure_ascii=False, indent=2))


{
  "model_name": "insectdex_yolo_candidate_20260606_000851",
  "model_version": "20260606_000851",
  "created_at": "2026-06-06T00:08:51.571553+00:00",
  "dataset_version_id": "ea6317a0-abb6-43fb-b795-db622a81cf10",
  "dataset_name": "insectdex_yolo_20260605_010113_49aee52e",
  "base_dataset_root": "/kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo",
  "start_model_path": "/kaggle/working/insectdex_auto_finetune/downloads/hf_model/production/best.pt",
  "unified_names": [
    "ant",
    "butterfly",
    "cockroach",
    "dragonfly",
    "fly",
    "grasshopper",
    "honeybee",
    "ladybug",
    "mosquito",
    "spider",
    "Cicada"
  ],
  "nc": 11,
  "demo_mode": true,
  "auto_promote": false,
  "epochs": 1,
  "img_size": 640,
  "batch_size": 8,
  "new_image_count": 2,
  "merged_train_count": 201,
  "merged_val_count": 51,
  "model_sha256": "061cf4d0112fd5b2d666c38607ba1a0f7f700650ddee06df91966014f49f7bf7",
  "metrics": {
    "epoch": 1.0,
    "time": 646.715,
    "tra

In [14]:
# ============================================================
# 14. Upload candidate model artifact
# ============================================================

def upload_to_huggingface(best_pt: Path, meta_path: Path, data_yaml_path: Path, model_name: str) -> Optional[str]:
    if not (HF_MODEL_REPO and HF_TOKEN and hf_api):
        return None

    candidate_dir = f"{HF_CANDIDATE_PREFIX}/{model_name}"
    print("Uploading candidate model to Hugging Face:", HF_MODEL_REPO, candidate_dir)

    hf_api.upload_file(
        path_or_fileobj=str(best_pt),
        path_in_repo=f"{candidate_dir}/best.pt",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )
    hf_api.upload_file(
        path_or_fileobj=str(meta_path),
        path_in_repo=f"{candidate_dir}/model_meta.json",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )
    hf_api.upload_file(
        path_or_fileobj=str(data_yaml_path),
        path_in_repo=f"{candidate_dir}/data.yaml",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )

    artifact_path = f"hf://{HF_MODEL_REPO}/{candidate_dir}/best.pt"
    return artifact_path


def upload_to_supabase_storage(best_pt: Path, meta_path: Path, data_yaml_path: Path, model_name: str) -> str:
    prefix = f"{MODEL_STORAGE_PREFIX}/{model_name}"

    files = [
        (best_pt, f"{prefix}/best.pt", "application/octet-stream"),
        (meta_path, f"{prefix}/model_meta.json", "application/json"),
        (data_yaml_path, f"{prefix}/data.yaml", "text/yaml"),
    ]

    for local_path, storage_path, content_type in files:
        print(f"Uploading to Supabase Storage: {MODEL_STORAGE_BUCKET}/{storage_path}")
        with open(local_path, "rb") as f:
            supabase.storage.from_(MODEL_STORAGE_BUCKET).upload(
                storage_path,
                f,
                file_options={"content-type": content_type, "upsert": "true"},
            )

    return f"supabase://{MODEL_STORAGE_BUCKET}/{prefix}/best.pt"


if TRAINING_SHOULD_RUN:
    artifact_path = upload_to_huggingface(
        best_pt=best_pt,
        meta_path=meta_path,
        data_yaml_path=MERGED_DATASET_DIR / "data.yaml",
        model_name=model_name,
    )

    if artifact_path is None:
        artifact_path = upload_to_supabase_storage(
            best_pt=best_pt,
            meta_path=meta_path,
            data_yaml_path=MERGED_DATASET_DIR / "data.yaml",
            model_name=model_name,
        )

    print("artifact_path:", artifact_path)


Uploading candidate model to Hugging Face: TOILAXIEN/insectdex-yolo-models candidates/insectdex_yolo_candidate_20260606_000851


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

artifact_path: hf://TOILAXIEN/insectdex-yolo-models/candidates/insectdex_yolo_candidate_20260606_000851/best.pt


In [15]:
# ============================================================
# 15. Optional: auto-promote to HF production path
# ============================================================

if TRAINING_SHOULD_RUN:
    production_promoted = False

    if AUTO_PROMOTE and HF_MODEL_REPO and HF_TOKEN and hf_api and not DEMO_MODE and new_image_count >= MIN_NEW_IMAGES_FOR_REAL_TRAIN:
        print("AUTO_PROMOTE enabled. Uploading candidate as production model.")
        hf_api.upload_file(
            path_or_fileobj=str(best_pt),
            path_in_repo=HF_PRODUCTION_PATH,
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            token=HF_TOKEN,
        )
        hf_api.upload_file(
            path_or_fileobj=str(meta_path),
            path_in_repo=HF_PRODUCTION_META_PATH,
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            token=HF_TOKEN,
        )
        hf_api.upload_file(
            path_or_fileobj=str(MERGED_DATASET_DIR / "data.yaml"),
            path_in_repo=HF_PRODUCTION_DATA_YAML_PATH,
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            token=HF_TOKEN,
        )
        production_promoted = True
    else:
        print("Skip production promote. Candidate model only.")
        print("Reasons may include: AUTO_PROMOTE=False, DEMO_MODE=True, small dataset, or missing HF config.")

    model_status = "production" if production_promoted else "candidate"


Skip production promote. Candidate model only.
Reasons may include: AUTO_PROMOTE=False, DEMO_MODE=True, small dataset, or missing HF config.


In [16]:
# ============================================================
# 16. Insert model_versions and update dataset_versions
# ============================================================

if TRAINING_SHOULD_RUN:
    model_versions_payload = {
        "name": model_name,
        "dataset_version_id": dataset_version["id"],
        "model_type": "yolo",
        "model_version": model_version,
        "metrics_json": metrics_json | {
            "demo_mode": DEMO_MODE,
            "new_image_count": new_image_count,
            "merged_train_count": merged_train_count,
            "merged_val_count": merged_val_count,
            "model_sha256": model_hash,
            "artifact_backend": "huggingface" if artifact_path.startswith("hf://") else "supabase_storage",
        },
        "artifact_path": artifact_path,
        "status": model_status,
        "deployed_at": now_utc_iso() if model_status == "production" else None,
    }

    print("Insert model_versions payload:")
    print(json.dumps(model_versions_payload, indent=2, ensure_ascii=False, default=str))

    response = supabase.table("model_versions").insert(model_versions_payload).execute()
    inserted_model_version = (response.data or [None])[0]
    if not inserted_model_version:
        raise RuntimeError("Insert model_versions failed")

    print("Inserted model_versions id:", inserted_model_version["id"])

    update_payload = {
        "status": "used_for_training",
    }
    supabase.table("dataset_versions").update(update_payload).eq("id", dataset_version["id"]).execute()
    print("Updated dataset_versions.status = used_for_training for", dataset_version["id"])


Insert model_versions payload:
{
  "name": "insectdex_yolo_candidate_20260606_000851",
  "dataset_version_id": "ea6317a0-abb6-43fb-b795-db622a81cf10",
  "model_type": "yolo",
  "model_version": "20260606_000851",
  "metrics_json": {
    "epoch": 1.0,
    "time": 646.715,
    "train/box_loss": 2.59641,
    "train/cls_loss": 6.57171,
    "train/dfl_loss": 2.77262,
    "metrics/precision(B)": 9e-05,
    "metrics/recall(B)": 0.07273,
    "metrics/mAP50(B)": 0.00082,
    "metrics/mAP50-95(B)": 0.00047,
    "val/box_loss": 2.94851,
    "val/cls_loss": 7.89376,
    "val/dfl_loss": 3.11487,
    "lr/pg0": 0.00016675,
    "lr/pg1": 0.00016675,
    "lr/pg2": 0.00016675,
    "demo_mode": true,
    "new_image_count": 2,
    "merged_train_count": 201,
    "merged_val_count": 51,
    "model_sha256": "061cf4d0112fd5b2d666c38607ba1a0f7f700650ddee06df91966014f49f7bf7",
    "artifact_backend": "huggingface"
  },
  "artifact_path": "hf://TOILAXIEN/insectdex-yolo-models/candidates/insectdex_yolo_candidate_

In [17]:
# ============================================================
# 17. Final summary
# ============================================================

if not TRAINING_SHOULD_RUN:
    print("DONE: no exported dataset found. Scheduled run completed safely.")
else:
    summary = {
        "dataset_version_id": dataset_version["id"],
        "model_name": model_name,
        "model_status": model_status,
        "artifact_path": artifact_path,
        "demo_mode": DEMO_MODE,
        "auto_promote": AUTO_PROMOTE,
        "new_image_count": new_image_count,
        "merged_train_count": merged_train_count,
        "merged_val_count": merged_val_count,
    }
    print("DONE: auto fine-tune completed.")
    print(json.dumps(summary, indent=2, ensure_ascii=False))


DONE: auto fine-tune completed.
{
  "dataset_version_id": "ea6317a0-abb6-43fb-b795-db622a81cf10",
  "model_name": "insectdex_yolo_candidate_20260606_000851",
  "model_status": "candidate",
  "artifact_path": "hf://TOILAXIEN/insectdex-yolo-models/candidates/insectdex_yolo_candidate_20260606_000851/best.pt",
  "demo_mode": true,
  "auto_promote": false,
  "new_image_count": 2,
  "merged_train_count": 201,
  "merged_val_count": 51
}
